# LeetCode 684 — Redundant Connection bằng Python

Link đề: https://leetcode.com/problems/redundant-connection/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Graph

Skeleton LeetCode:

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đề cho biết "nếu có nhiều đáp án, trả về cạnh **xuất hiện sau cùng** trong `edges`", nên đáp án thực ra **duy nhất**. Dù vậy ở phần test ta kiểm bằng **hàm kiểm tra tính hợp lệ** (loại cạnh đó thì còn lại một cây, và nó là cạnh cuối cùng thỏa điều kiện) thay vì chỉ so với một đáp án cứng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một đồ thị vô hướng vốn là **cây** `n` nút rồi bị **thêm đúng một cạnh**, hãy tìm cạnh có thể **xóa đi để lại một cây**; nếu có nhiều cạnh xóa được thì chọn cạnh đứng **sau cùng** trong danh sách.
2. Nắm các công cụ Python: danh sách kề (`adjacency list`), duyệt **DFS/BFS** bằng ngăn xếp / `deque`, cấu trúc **Disjoint Set Union (Union-Find)** với nén đường đi, đếm **bậc (degree)** của nút, `set` để kiểm tra "nút thuộc chu trình".
3. Đi từ **thử xóa từng cạnh từ cuối và kiểm tra còn là cây** → **thêm cạnh dần, kiểm tra hai đầu đã liên thông chưa bằng DFS** → **Union-Find** → **cắt dần các lá để tìm chu trình** → **DFS tìm chu trình theo cha**.
4. Hiểu mấu chốt: cây có `n` nút thì có đúng `n - 1` cạnh; thêm một cạnh tạo **đúng một chu trình**; **mọi cạnh trên chu trình** đều xóa được, và "cạnh đầu tiên khiến hai đầu **đã liên thông**" khi duyệt theo thứ tự đầu vào chính là **cạnh sau cùng trong chu trình**.
5. Tránh các lỗi kinh điển: trả cạnh **đầu tiên** của chu trình thay vì cạnh **sau cùng**, so sánh `p[a] == p[b]` mà quên tìm **gốc** (`find`), tạo mảng cha thiếu một phần tử vì nút đánh số từ `1`, và nhầm "bậc `>= 2`" với "nằm trên chu trình".

## 2. Đề bài & ví dụ

Trong bài này, **cây** là đồ thị **vô hướng**, **liên thông** và **không có chu trình**.

Bạn được cho một đồ thị ban đầu là cây có `n` nút đánh số từ `1` đến `n`, sau đó **thêm một cạnh nữa**. Cạnh thêm vào nối **hai đỉnh khác nhau** từ `1` đến `n` và **chưa từng tồn tại**. Đồ thị được cho dưới dạng mảng `edges` độ dài `n`, trong đó `edges[i] = [a_i, b_i]` nghĩa là có cạnh nối `a_i` và `b_i`.

Hãy trả về **một cạnh** có thể xóa để đồ thị còn lại là một **cây `n` nút**. Nếu có nhiều đáp án, trả về cạnh **xuất hiện sau cùng** trong đầu vào.

**Ví dụ 1**

- Input: `edges = [[1,2],[1,3],[2,3]]`
- Output: `[2,3]`
- Giải thích: ba cạnh tạo thành một tam giác; xóa cạnh nào cũng còn cây, ta chọn cạnh cuối `[2,3]`.

**Ví dụ 2**

- Input: `edges = [[1,2],[2,3],[3,4],[1,4],[1,5]]`
- Output: `[1,4]`
- Giải thích: chu trình là `1-2-3-4-1` gồm các cạnh `[1,2],[2,3],[3,4],[1,4]`; cạnh `[1,5]` không nằm trên chu trình nên **không** xóa được. Trong các cạnh xóa được, `[1,4]` xuất hiện sau cùng.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `edges = [[1,2],[2,3],[1,3]]` → Output: `[1,3]`

**Ràng buộc:**

- `n == edges.length`, `3 <= n <= 1000`.
- `edges[i].length == 2`, `1 <= a_i < b_i <= edges.length`, `a_i != b_i`.
- Không có cạnh lặp; đồ thị đã cho **liên thông**.

Chú ý: vì đồ thị liên thông với `n` nút và `n` cạnh nên nó có **đúng một chu trình**.

In [1]:
from typing import List
from collections import deque, defaultdict
import random
import sys
import time

examples = [
    ([[1, 2], [1, 3], [2, 3]], [2, 3]),
    ([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]], [1, 4]),
    ([[1, 2], [2, 3], [1, 3]], [1, 3]),
]
for edges, expected in examples:
    print(edges, "-> kỳ vọng", expected)

[[1, 2], [1, 3], [2, 3]] -> kỳ vọng [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] -> kỳ vọng [1, 4]
[[1, 2], [2, 3], [1, 3]] -> kỳ vọng [1, 3]


## 3. Kiến thức nền

### 3.1. Cây có `n` nút thì có đúng `n - 1` cạnh

Một đồ thị vô hướng với `n` nút là **cây** khi và chỉ khi nó **liên thông** và có **đúng `n - 1` cạnh** (khi đó tự động không có chu trình). Đầu vào bài này có `n` cạnh — **thừa đúng một cạnh** so với cây — nên có **đúng một chu trình**. Xóa một cạnh **thuộc chu trình** thì còn `n - 1` cạnh mà vẫn liên thông ⇒ là cây. Xóa một cạnh **không** thuộc chu trình (cầu) thì mất liên thông.

In [2]:
def is_tree(n, edges):
    """Cây n nút (đánh số 1..n): đủ n-1 cạnh và liên thông (BFS từ nút 1)."""
    if len(edges) != n - 1:
        return False
    adj = [[] for _ in range(n + 1)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    seen = {1}
    queue = deque([1])
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                queue.append(v)
    return len(seen) == n

edges = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
print("đủ n cạnh, là cây?", is_tree(5, edges))
for i, e in enumerate(edges):
    rest = edges[:i] + edges[i + 1:]
    print(f"xóa {e}: còn là cây? {is_tree(5, rest)}")

đủ n cạnh, là cây? False
xóa [1, 2]: còn là cây? True
xóa [2, 3]: còn là cây? True
xóa [3, 4]: còn là cây? True
xóa [1, 4]: còn là cây? True
xóa [1, 5]: còn là cây? False


### 3.2. Danh sách kề (adjacency list)

Đồ thị vô hướng lưu bằng `adj[u]` = danh sách nút kề `u`. Mỗi cạnh `[a, b]` thêm **cả hai chiều**: `adj[a].append(b)` và `adj[b].append(a)`. Vì nút đánh số từ `1` nên ta cấp `n + 1` ô (bỏ ô `0`), hoặc dùng `defaultdict(list)`.

In [3]:
edges = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
n = len(edges)
adj = [[] for _ in range(n + 1)]
for a, b in edges:
    adj[a].append(b)
    adj[b].append(a)
for u in range(1, n + 1):
    print(f"adj[{u}] = {adj[u]}")

g = defaultdict(list)
for a, b in edges:
    g[a].append(b)
    g[b].append(a)
print("defaultdict:", dict(g))

adj[1] = [2, 4, 5]
adj[2] = [1, 3]
adj[3] = [2, 4]
adj[4] = [3, 1]
adj[5] = [1]
defaultdict: {1: [2, 4, 5], 2: [1, 3], 3: [2, 4], 4: [3, 1], 5: [1]}


### 3.3. DFS / BFS và câu hỏi "có đường đi từ `a` tới `b` không?"

Duyệt đồ thị từ `a` bằng **ngăn xếp** (DFS) hay **hàng đợi** (BFS), đánh dấu nút đã thăm; nếu gặp `b` thì có đường đi, tức `a` và `b` **cùng một thành phần liên thông**. Ta viết bản lặp (không đệ quy) để an toàn với đồ thị sâu.

In [4]:
def has_path(adj, a, b):
    stack = [a]
    seen = {a}
    while stack:
        u = stack.pop()
        if u == b:
            return True
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return False

print(has_path(adj, 1, 4), has_path(adj, 5, 3), has_path(adj, 2, 5))
small = [[], [2], [1], []]           # nút 1-2 nối nhau, nút 3 cô lập
print(has_path(small, 1, 2), has_path(small, 1, 3))

True True True
True False


### 3.4. Union-Find (Disjoint Set Union)

Union-Find quản lý các **tập rời nhau**. Mỗi nút trỏ tới một "cha"; nút trỏ vào chính nó là **gốc** (đại diện tập). Hai thao tác:

- `find(x)`: đi theo cha cho tới gốc; **nén đường đi** (`p[x] = find(p[x])`) để lần sau nhanh.
- `union(a, b)`: tìm gốc của hai nút; nếu **khác** nhau thì nối một gốc vào gốc kia; nếu **giống nhau** thì hai nút **đã cùng tập** (nối thêm sẽ tạo chu trình).

Với đồ thị, cạnh `[a, b]` mà `find(a) == find(b)` nghĩa là `a` và `b` đã liên thông qua các cạnh trước đó ⇒ cạnh này khép thành chu trình.

In [5]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n + 1))          # nút 1..n, bỏ ô 0

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                     # đã cùng tập
        self.p[ra] = rb
        return True

d = DSU(5)
print(d.union(1, 2), d.union(2, 3), d.union(1, 3), "<- cạnh (1,3) khép chu trình")
print("cha:", d.p, "| find(1) =", d.find(1), "| find(3) =", d.find(3))

True True False <- cạnh (1,3) khép chu trình
cha: [0, 3, 3, 3, 4, 5] | find(1) = 3 | find(3) = 3


### 3.5. Bậc (degree) của nút và cắt dần các lá

**Bậc** của một nút là số cạnh gắn với nó. Một **lá** là nút bậc `1`. Nếu liên tục **gỡ các lá** (và giảm bậc láng giềng của chúng), thì trong một đồ thị liên thông có đúng một chu trình, mọi phần "cành cây" sẽ bị gỡ hết và **chỉ còn lại các nút của chu trình** (mỗi nút chu trình luôn giữ bậc `>= 2`). Ta dùng `deque` làm hàng đợi các lá.

In [6]:
edges = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
n = len(edges)
deg = [0] * (n + 1)
adj = [[] for _ in range(n + 1)]
for a, b in edges:
    deg[a] += 1
    deg[b] += 1
    adj[a].append(b)
    adj[b].append(a)
print("bậc:", deg[1:])

removed = set()
queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
while queue:
    u = queue.popleft()
    removed.add(u)
    for v in adj[u]:
        if v not in removed:
            deg[v] -= 1
            if deg[v] == 1:
                queue.append(v)
print("các lá đã gỡ:", sorted(removed), "| nút còn lại (chu trình):", [u for u in range(1, n + 1) if u not in removed])

bậc: [3, 2, 2, 2, 1]
các lá đã gỡ: [5] | nút còn lại (chu trình): [1, 2, 3, 4]


### 3.6. Nút đánh số từ 1 và cái bẫy kích thước mảng

Nút là `1..n`, nên mảng chỉ số trực tiếp theo nút cần **`n + 1` phần tử** (ô `0` bỏ phí). Tạo `list(range(n))` chỉ có chỉ số `0..n-1` ⇒ truy cập `p[n]` gây `IndexError`. Hai cách an toàn: kích thước `n + 1`, hoặc trừ 1 khi đọc (`a - 1`) như lời giải mẫu.

In [7]:
n = 5
p_bad = list(range(n))
try:
    print(p_bad[n])
except IndexError as e:
    print("IndexError:", e)
p_ok = list(range(n + 1))
print("p_ok[n] =", p_ok[n])

IndexError: list index out of range
p_ok[n] = 5


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `edges = [[1,2],[2,3],[3,4],[1,4],[1,5]]`. Hình dung: dựng đồ thị **từng cạnh một theo thứ tự**.

1. Thêm `1-2`: đồ thị `{1-2}`. Hai nút chưa liên thông trước đó ⇒ OK.
2. Thêm `2-3`: `3` chưa nối với `2` ⇒ OK. Đồ thị là đường `1-2-3`.
3. Thêm `3-4`: OK. Đường `1-2-3-4`.
4. Thêm `1-4`: **`1` và `4` đã liên thông** (qua `1-2-3-4`) ⇒ cạnh này khép thành chu trình `1-2-3-4-1`. **Đây là cạnh thừa.**
5. Thêm `1-5` sẽ không bao giờ khép chu trình mới (đồ thị chỉ có một chu trình).

Ta dừng ở bước 4 và trả `[1,4]` ✓.

**Vì sao cạnh đầu tiên khép chu trình là cạnh "sau cùng"?** Chu trình duy nhất gồm các cạnh `[1,2],[2,3],[3,4],[1,4]`. Chu trình chỉ **hoàn tất** khi cạnh cuối cùng (theo thứ tự đầu vào) của nó xuất hiện. Cạnh khép chu trình lần đầu chính là cạnh có chỉ số lớn nhất trong chu trình — đúng yêu cầu "xuất hiện sau cùng" trong các cạnh có thể xóa.

Còn cạnh `[1,5]` đứng **sau** `[1,4]` trong danh sách nhưng **không xóa được** (xóa nó thì nút `5` bị cô lập). Vì vậy đáp án là cạnh sau cùng **trong chu trình**, không phải cạnh sau cùng của cả danh sách. Hãy kiểm bằng code: với mỗi cạnh, xóa đi rồi xem còn là cây không.

In [8]:
def removable_indices(edges):
    n = len(edges)
    return [i for i in range(n) if is_tree(n, edges[:i] + edges[i + 1:])]

for edges, expected in examples:
    idx = removable_indices(edges)
    print(edges)
    print("  các cạnh xóa được:", [edges[i] for i in idx], "| cạnh sau cùng trong số đó:", edges[idx[-1]], "| kỳ vọng:", expected)

[[1, 2], [1, 3], [2, 3]]
  các cạnh xóa được: [[1, 2], [1, 3], [2, 3]] | cạnh sau cùng trong số đó: [2, 3] | kỳ vọng: [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
  các cạnh xóa được: [[1, 2], [2, 3], [3, 4], [1, 4]] | cạnh sau cùng trong số đó: [1, 4] | kỳ vọng: [1, 4]
[[1, 2], [2, 3], [1, 3]]
  các cạnh xóa được: [[1, 2], [2, 3], [1, 3]] | cạnh sau cùng trong số đó: [1, 3] | kỳ vọng: [1, 3]


## 5. Cách 1 — Brute Force: thử xóa từng cạnh từ cuối

**Ý tưởng:** làm đúng theo định nghĩa. Duyệt `i` từ `n - 1` về `0` (**từ cuối lên**, vì cần cạnh xuất hiện sau cùng). Với mỗi `i`, **bỏ cạnh `edges[i]`** rồi kiểm tra đồ thị còn lại có là **cây** không (liên thông, đủ `n - 1` cạnh). Cạnh đầu tiên (từ cuối) thỏa thì trả về.

Chắc chắn đúng, bám sát đề, nên ta dùng làm **oracle**.

In [9]:
def find_redundant_bruteforce(edges: List[List[int]]) -> List[int]:
    n = len(edges)
    for i in range(n - 1, -1, -1):
        rest = edges[:i] + edges[i + 1:]
        if is_tree(n, rest):
            return edges[i]

for edges, expected in examples:
    got = find_redundant_bruteforce(edges)
    print(edges, got, "OK" if got == expected else "SAI")

[[1, 2], [1, 3], [2, 3]] [2, 3] OK
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] [1, 4] OK
[[1, 2], [2, 3], [1, 3]] [1, 3] OK


**Độ phức tạp:** tối đa `n` lần thử, mỗi lần dựng đồ thị và BFS `O(n)` ⇒ `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n = 1000` ⇒ khoảng `10^6` phép — còn chạy được trong Python, nhưng chậm và lặp lại việc dựng đồ thị `n` lần.

**Nút thắt:** mỗi lần ta **làm lại từ đầu** việc xét tính liên thông. Ta muốn một cấu trúc **cập nhật dần** khi thêm cạnh, biết ngay hai nút đã liên thông chưa.

In [10]:
rng = random.Random(0)

def random_graph(rng, n):
    """Cây ngẫu nhiên n nút (cha ngẫu nhiên) + 1 cạnh thêm; nhãn nút và thứ tự cạnh được xáo; a < b."""
    labels = list(range(1, n + 1))
    rng.shuffle(labels)
    edge_set = set()
    for v in range(1, n):
        u = rng.randrange(v)
        a, b = labels[u], labels[v]
        edge_set.add((min(a, b), max(a, b)))
    while True:
        a, b = rng.sample(range(1, n + 1), 2)
        e = (min(a, b), max(a, b))
        if e not in edge_set:
            edge_set.add(e)
            break
    edges = [list(e) for e in edge_set]
    rng.shuffle(edges)
    return edges

for n in [100, 200, 400, 800]:
    edges = random_graph(rng, n)
    t0 = time.perf_counter()
    find_redundant_bruteforce(edges)
    dt = time.perf_counter() - t0
    print(f"n = {n:4d}: brute force {dt * 1000:8.1f} ms")

n =  100: brute force      0.1 ms
n =  200: brute force      0.2 ms
n =  400: brute force      1.7 ms
n =  800: brute force      0.3 ms


## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá: **thêm các cạnh theo thứ tự đầu vào và theo dõi các thành phần liên thông**. Cạnh `[a, b]` mà `a` và `b` **đã cùng một thành phần** là cạnh khép chu trình lần đầu — và (như đã lập luận ở mục 4) chính là đáp án.

Muốn hỏi "`a` và `b` đã cùng thành phần chưa?" **nhanh**, ta dùng **Union-Find**:

```
p = [0, 1, 2, ..., n]            # mỗi nút là một tập riêng
for a, b in edges:
    ra, rb = find(a), find(b)
    if ra == rb:                  # đã cùng tập -> cạnh này khép chu trình
        return [a, b]
    p[ra] = rb                    # hợp hai tập
```

Mỗi cạnh tốn gần `O(1)` (độ phức tạp trung bình `O(α(n))` với nén đường đi). Cả đồ thị: gần `O(n)`.

Hãy tính tay trên ví dụ 2 với mảng cha (nối `ra` vào `rb`):

| cạnh | `find(a)` | `find(b)` | cùng tập? | mảng `p[1..5]` sau cạnh |
|---|---|---|---|---|
| `[1,2]` | 1 | 2 | không | `[2,2,3,4,5]` |
| `[2,3]` | 2 | 3 | không | `[2,3,3,4,5]` |
| `[3,4]` | 3 | 4 | không | `[2,3,4,4,5]` |
| `[1,4]` | 4 (1→2→3→4) | 4 | **có** | trả `[1,4]` |

## 7. Các cách tiếp theo

Cùng một đích (tìm cạnh khép chu trình sau cùng), ta có nhiều cách nhìn: **Union-Find**, **DFS kiểm tra đường đi trước khi thêm cạnh**, **cắt dần các lá để lộ chu trình**, và **DFS tìm chu trình theo cha**. Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Union-Find

**Giải thích từng đoạn:**

1. `p = list(range(n + 1))` — mỗi nút là một tập; cấp `n + 1` ô vì nút đánh số từ `1`.
2. `find(x)` với **nén đường đi** để cây tập nông.
3. Duyệt cạnh theo thứ tự: nếu hai đầu có cùng gốc thì trả cạnh đó; ngược lại nối hai gốc.

In [11]:
class SolutionUF:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]               # nén đường đi (path halving), bản lặp
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:
                return [a, b]
            p[ra] = rb
        return []

for edges, expected in examples:
    print(edges, SolutionUF().findRedundantConnection(edges), "kỳ vọng", expected)

[[1, 2], [1, 3], [2, 3]] [2, 3] kỳ vọng [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] [1, 4] kỳ vọng [1, 4]
[[1, 2], [2, 3], [1, 3]] [1, 3] kỳ vọng [1, 3]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:
                return [a, b]
            p[ra] = rb
        return []
```

**Độ phức tạp:** thời gian gần tuyến tính `O(n · α(n))` (với nén đường đi, `α` là hàm Ackermann ngược, coi như hằng số), bộ nhớ `O(n)`. Bản `find` viết bằng **vòng lặp** tránh đệ quy sâu.

### 7.2. Cách 3 — Thêm cạnh dần, DFS kiểm tra "đã có đường đi chưa"

**Ý tưởng:** cùng thứ tự duyệt như Union-Find, nhưng thay vì cấu trúc tập, ta **giữ đồ thị đang xây** (danh sách kề) và **trước khi thêm** cạnh `[a, b]`, dùng DFS kiểm tra `b` có tới được từ `a` trong đồ thị hiện tại không. Nếu có ⇒ thêm cạnh sẽ khép chu trình ⇒ trả cạnh đó. Chậm hơn Union-Find (mỗi lần DFS `O(n)`) nhưng **rất dễ hiểu**.

In [12]:
class SolutionDFS:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]

        def connected(a: int, b: int) -> bool:
            stack = [a]
            seen = {a}
            while stack:
                u = stack.pop()
                if u == b:
                    return True
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            return False

        for a, b in edges:
            if connected(a, b):
                return [a, b]
            adj[a].append(b)
            adj[b].append(a)
        return []

for edges, expected in examples:
    print(edges, SolutionDFS().findRedundantConnection(edges), "kỳ vọng", expected)

[[1, 2], [1, 3], [2, 3]] [2, 3] kỳ vọng [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] [1, 4] kỳ vọng [1, 4]
[[1, 2], [2, 3], [1, 3]] [1, 3] kỳ vọng [1, 3]


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]

        def connected(a: int, b: int) -> bool:
            stack = [a]
            seen = {a}
            while stack:
                u = stack.pop()
                if u == b:
                    return True
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            return False

        for a, b in edges:
            if connected(a, b):
                return [a, b]
            adj[a].append(b)
            adj[b].append(a)
        return []
```

**Độ phức tạp:** mỗi cạnh một lần DFS `O(n)` ⇒ `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n ≤ 1000` vẫn đủ nhanh.

### 7.3. Cách 4 — Cắt dần các lá để lộ chu trình

**Ý tưởng khác hẳn:** không cần xét thứ tự cạnh khi tìm chu trình. Dựng **toàn bộ đồ thị**, đếm **bậc**, rồi **gỡ lần lượt các nút bậc `1`** (lá) và giảm bậc láng giềng. Cuối cùng các nút **còn lại** chính là các nút của **chu trình duy nhất** (xem mục 3.5). Một cạnh xóa được ⇔ **cả hai đầu đều nằm trên chu trình** (đồ thị chỉ có đúng một chu trình). Chọn cạnh **sau cùng** trong `edges` thỏa điều đó.

**Giải thích từng đoạn:**

1. Dựng `adj` và `deg`.
2. `queue` chứa mọi nút bậc `1`. Lấy `u`, đánh dấu gỡ; với mỗi láng giềng `v` chưa gỡ: `deg[v] -= 1`, nếu thành `1` thì đưa vào `queue`.
3. `cycle` = tập nút chưa bị gỡ.
4. Duyệt `edges` **từ cuối lên**, trả cạnh đầu tiên có cả hai đầu thuộc `cycle`.

In [13]:
class SolutionPeel:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]
        deg = [0] * (n + 1)
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
            deg[a] += 1
            deg[b] += 1

        removed = [False] * (n + 1)
        queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
        while queue:
            u = queue.popleft()
            removed[u] = True
            for v in adj[u]:
                if not removed[v]:
                    deg[v] -= 1
                    if deg[v] == 1:
                        queue.append(v)

        for a, b in reversed(edges):
            if not removed[a] and not removed[b]:
                return [a, b]
        return []

for edges, expected in examples:
    print(edges, SolutionPeel().findRedundantConnection(edges), "kỳ vọng", expected)

[[1, 2], [1, 3], [2, 3]] [2, 3] kỳ vọng [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] [1, 4] kỳ vọng [1, 4]
[[1, 2], [2, 3], [1, 3]] [1, 3] kỳ vọng [1, 3]


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]
        deg = [0] * (n + 1)
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
            deg[a] += 1
            deg[b] += 1

        removed = [False] * (n + 1)
        queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
        while queue:
            u = queue.popleft()
            removed[u] = True
            for v in adj[u]:
                if not removed[v]:
                    deg[v] -= 1
                    if deg[v] == 1:
                        queue.append(v)

        for a, b in reversed(edges):
            if not removed[a] and not removed[b]:
                return [a, b]
        return []
```

**Độ phức tạp:** mỗi nút gỡ một lần, mỗi cạnh xét hai lần ⇒ `O(n)` thời gian, `O(n)` bộ nhớ.

Một chi tiết quan trọng: ta **không** được kết luận "cạnh nào có cả hai đầu bậc `>= 2` thì xóa được" bằng bậc ban đầu — đó là lỗi hay gặp ở mục 13. Phải **cắt lá lặp đi lặp lại** cho tới khi chỉ còn chu trình.

### 7.4. Cách 5 — DFS tìm chu trình bằng con trỏ cha

**Ý tưởng:** dựng toàn bộ đồ thị rồi **DFS từ nút 1** ghi lại `parent` và `depth`. Trong DFS vô hướng, một cạnh tới nút **đã thăm** mà **không phải cha** là **cạnh ngược (back edge)** nối `u` với một **tổ tiên** `v` (độ sâu nhỏ hơn). Chu trình là đường từ `u` leo theo `parent` lên tới `v`, rồi cạnh ngược quay lại `u`. Gom các nút trên chu trình vào `cycle`; đáp án là cạnh **sau cùng** trong `edges` có cả hai đầu thuộc `cycle`.

**Giải thích từng đoạn:**

1. DFS lặp có **ngăn xếp các iterator** để đúng thứ tự DFS thật (tổ tiên – hậu duệ): mỗi phần tử ngăn xếp là `(nút, iterator hàng xóm)`.
2. Khi duyệt hàng xóm `v` của `u`: nếu `v` chưa thăm → đặt `parent[v] = u`, `depth[v] = depth[u] + 1`, đẩy vào ngăn xếp; nếu `v` đã thăm, `v != parent[u]` và `depth[v] < depth[u]` → tìm được cạnh ngược.
3. Từ `u` leo `parent` lên tới `v` để gom `cycle`.

In [14]:
class SolutionCycleDFS:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)

        parent = [0] * (n + 1)
        depth = [-1] * (n + 1)
        depth[1] = 0
        stack = [(1, iter(adj[1]))]
        cycle = set()
        while stack and not cycle:
            u, it = stack[-1]
            advanced = False
            for v in it:
                if depth[v] == -1:                       # nút mới: đi sâu xuống
                    parent[v] = u
                    depth[v] = depth[u] + 1
                    stack.append((v, iter(adj[v])))
                    advanced = True
                    break
                if v != parent[u] and depth[v] < depth[u]:   # cạnh ngược tới tổ tiên v
                    x = u
                    cycle.add(v)
                    while x != v:
                        cycle.add(x)
                        x = parent[x]
                    break
            if cycle:
                break
            if not advanced:
                stack.pop()

        for a, b in reversed(edges):
            if a in cycle and b in cycle:
                return [a, b]
        return []

for edges, expected in examples:
    print(edges, SolutionCycleDFS().findRedundantConnection(edges), "kỳ vọng", expected)

[[1, 2], [1, 3], [2, 3]] [2, 3] kỳ vọng [2, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] [1, 4] kỳ vọng [1, 4]
[[1, 2], [2, 3], [1, 3]] [1, 3] kỳ vọng [1, 3]


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)

        parent = [0] * (n + 1)
        depth = [-1] * (n + 1)
        depth[1] = 0
        stack = [(1, iter(adj[1]))]
        cycle = set()
        while stack and not cycle:
            u, it = stack[-1]
            advanced = False
            for v in it:
                if depth[v] == -1:
                    parent[v] = u
                    depth[v] = depth[u] + 1
                    stack.append((v, iter(adj[v])))
                    advanced = True
                    break
                if v != parent[u] and depth[v] < depth[u]:
                    x = u
                    cycle.add(v)
                    while x != v:
                        cycle.add(x)
                        x = parent[x]
                    break
            if cycle:
                break
            if not advanced:
                stack.pop()

        for a, b in reversed(edges):
            if a in cycle and b in cycle:
                return [a, b]
        return []
```

**Độ phức tạp:** mỗi nút và mỗi cạnh được xét `O(1)` lần ⇒ `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Ta theo dõi Union-Find (cách 2) trên ví dụ 2 `[[1,2],[2,3],[3,4],[1,4],[1,5]]`: với mỗi cạnh in `find` của hai đầu, kết luận và mảng cha `p[1..5]`.

Bảng trạng thái (xem mục 6): sau `[1,2]` → `p = [2,2,3,4,5]`; sau `[2,3]` → `[2,3,3,4,5]`; sau `[3,4]` → `[2,3,4,4,5]`; tới `[1,4]`: `find(1)` đi `1→2→3→4` được `4`, `find(4) = 4` ⇒ cùng tập ⇒ trả `[1,4]`.

In [15]:
def trace_union_find(edges):
    n = len(edges)
    p = list(range(n + 1))

    def find(x):
        path = [x]
        while p[x] != x:
            x = p[x]
            path.append(x)
        return x, path

    for a, b in edges:
        ra, pa = find(a)
        rb, pb = find(b)
        print(f"cạnh {[a, b]}: find({a}) = {ra} (đường {pa}), find({b}) = {rb} (đường {pb})")
        if ra == rb:
            print(f"  -> cùng tập: cạnh {[a, b]} khép chu trình, trả về")
            return [a, b]
        p[ra] = rb
        print(f"  -> khác tập: nối gốc {ra} vào gốc {rb}; p[1..{n}] = {p[1:]}")
    return []

print("KẾT QUẢ:", trace_union_find([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

cạnh [1, 2]: find(1) = 1 (đường [1]), find(2) = 2 (đường [2])
  -> khác tập: nối gốc 1 vào gốc 2; p[1..5] = [2, 2, 3, 4, 5]
cạnh [2, 3]: find(2) = 2 (đường [2]), find(3) = 3 (đường [3])
  -> khác tập: nối gốc 2 vào gốc 3; p[1..5] = [2, 3, 3, 4, 5]
cạnh [3, 4]: find(3) = 3 (đường [3]), find(4) = 4 (đường [4])
  -> khác tập: nối gốc 3 vào gốc 4; p[1..5] = [2, 3, 4, 4, 5]
cạnh [1, 4]: find(1) = 4 (đường [1, 2, 3, 4]), find(4) = 4 (đường [4])
  -> cùng tập: cạnh [1, 4] khép chu trình, trả về
KẾT QUẢ: [1, 4]


Giờ trace cách 4 (cắt lá) trên cùng ví dụ: in bậc ban đầu, từng lá được gỡ, bậc mới của láng giềng, và tập nút còn lại.

In [16]:
def trace_peel(edges):
    n = len(edges)
    adj = [[] for _ in range(n + 1)]
    deg = [0] * (n + 1)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
        deg[a] += 1
        deg[b] += 1
    print("bậc ban đầu deg[1..n] =", deg[1:])
    removed = [False] * (n + 1)
    queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
    print("hàng đợi lá ban đầu:", list(queue))
    while queue:
        u = queue.popleft()
        removed[u] = True
        print(f"gỡ lá {u}")
        for v in adj[u]:
            if not removed[v]:
                deg[v] -= 1
                print(f"  láng giềng {v}: bậc giảm còn {deg[v]}", "-> thành lá, đưa vào hàng đợi" if deg[v] == 1 else "")
                if deg[v] == 1:
                    queue.append(v)
    cycle = [u for u in range(1, n + 1) if not removed[u]]
    print("các nút còn lại (chu trình):", cycle)
    for a, b in reversed(edges):
        print(f"xét cạnh {[a, b]} từ cuối: hai đầu đều thuộc chu trình? {a in cycle and b in cycle}")
        if a in cycle and b in cycle:
            return [a, b]

print("KẾT QUẢ:", trace_peel([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

bậc ban đầu deg[1..n] = [3, 2, 2, 2, 1]
hàng đợi lá ban đầu: [5]
gỡ lá 5
  láng giềng 1: bậc giảm còn 2 
các nút còn lại (chu trình): [1, 2, 3, 4]
xét cạnh [1, 5] từ cuối: hai đầu đều thuộc chu trình? False
xét cạnh [1, 4] từ cuối: hai đầu đều thuộc chu trình? True
KẾT QUẢ: [1, 4]


## 9. Code nộp LeetCode

Ta chọn **Cách 2 (Union-Find)**: ngắn, nhanh gần tuyến tính, và đúng ngay theo thứ tự đầu vào nên **tự động** cho cạnh sau cùng. Class `Solution` đúng chữ ký:

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:
                return [a, b]
            p[ra] = rb
        return []
```

In [17]:
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:
                return [a, b]
            p[ra] = rb
        return []

for edges, expected in examples:
    got = Solution().findRedundantConnection(edges)
    assert got == expected, (edges, got, expected)
    print(edges, "->", got, "OK")

[[1, 2], [1, 3], [2, 3]] -> [2, 3] OK
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] -> [1, 4] OK
[[1, 2], [2, 3], [1, 3]] -> [1, 3] OK


## 10. Vì sao đúng

**Khẳng định 1: đồ thị đã cho có đúng một chu trình.** Cây `n` nút có `n - 1` cạnh, liên thông. Thêm một cạnh nối hai nút **đã liên thông** (trong cây mọi cặp nút đều liên thông) tạo **đúng một** chu trình: chu trình gồm cạnh mới và đường đi **duy nhất** trong cây giữa hai đầu của nó.

**Khẳng định 2: một cạnh xóa được ⇔ nó nằm trên chu trình.** Xóa một cạnh trên chu trình: đồ thị vẫn liên thông (đi vòng phần còn lại của chu trình) và còn `n - 1` cạnh ⇒ là cây. Xóa cạnh ngoài chu trình: cạnh đó là **cầu**, xóa đi làm mất liên thông ⇒ không còn là cây.

**Khẳng định 3: Union-Find trả đúng cạnh sau cùng của chu trình.** Gọi `C` là tập cạnh của chu trình, `e*` là cạnh có chỉ số lớn nhất trong `C`. Duyệt cạnh theo thứ tự: trước khi gặp `e*`, các cạnh đã thêm **không chứa đủ** chu trình nên chúng tạo thành một **rừng** (không có chu trình) — với mọi cạnh trong rừng, hai đầu **chưa** cùng tập lúc được thêm (nếu đã cùng tập thì cạnh đó khép một chu trình — mâu thuẫn với việc chu trình duy nhất cần `e*`). Khi gặp `e*`: hai đầu của nó đã nối với nhau bằng phần còn lại của `C` (đều có chỉ số nhỏ hơn) ⇒ `find` bằng nhau ⇒ thuật toán trả `e*`. Mà `e*` chính là cạnh **sau cùng** trong các cạnh xóa được (các cạnh xóa được = `C`).

**Kiểm chứng bằng code:** ta viết **hàm kiểm tra tính hợp lệ** `is_valid_answer(edges, ans)` — (1) `ans` thuộc `edges`; (2) xóa `ans` thì còn lại **một cây**; (3) mọi cạnh xuất hiện **sau** `ans` đều **không** xóa được. Rồi kiểm cả năm cách trên nhiều đồ thị ngẫu nhiên.

In [18]:
def is_valid_answer(edges, ans):
    """True nếu ans là cạnh xóa được và là cạnh xóa được xuất hiện SAU CÙNG trong edges."""
    n = len(edges)
    if list(ans) not in edges:
        return False
    i = edges.index(list(ans))
    if not is_tree(n, edges[:i] + edges[i + 1:]):
        return False                                   # (2) xóa ans không còn là cây
    for j in range(i + 1, n):                          # (3) các cạnh sau ans phải không xóa được
        if is_tree(n, edges[:j] + edges[j + 1:]):
            return False
    return True

for edges, expected in examples:
    assert is_valid_answer(edges, expected)
    assert not is_valid_answer(edges, edges[0]) or edges[0] == expected      # cạnh đầu thường không phải đáp án sau cùng
print("hàm kiểm tra hợp lệ: OK trên các ví dụ")
print(is_valid_answer([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]], [1, 2]), "<- [1,2] xóa được nhưng không phải cạnh sau cùng")
print(is_valid_answer([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]], [1, 5]), "<- [1,5] không xóa được (cầu)")

hàm kiểm tra hợp lệ: OK trên các ví dụ
False <- [1,2] xóa được nhưng không phải cạnh sau cùng
False <- [1,5] không xóa được (cầu)


**Kết thúc:** Union-Find duyệt đúng `n` cạnh; theo khẳng định 3 nó luôn dừng khi gặp `e*` (tồn tại vì đồ thị có chu trình).

## 11. Time / Space Complexity

Gọi `n` là số nút (= số cạnh).

| Cách | Time | Space |
|---|---|---|
| 1. Thử xóa từng cạnh + BFS | `O(n²)` | `O(n)` |
| 2. Union-Find | `O(n · α(n))` ≈ `O(n)` | `O(n)` |
| 3. Thêm dần + DFS kiểm tra đường đi | `O(n²)` | `O(n)` |
| 4. Cắt lá | `O(n)` | `O(n)` |
| 5. DFS tìm chu trình theo cha | `O(n)` | `O(n)` |

Ở `n = 1000`, `n²` = `10^6` vẫn nhỏ; sự khác biệt là bài học về **cách cập nhật dần** (Union-Find) so với **làm lại từ đầu**. Đo thực tế trên đồ thị ngẫu nhiên kích thước tăng dần.

In [19]:
impls = {
    "uf": lambda e: SolutionUF().findRedundantConnection(e),
    "dfs": lambda e: SolutionDFS().findRedundantConnection(e),
    "peel": lambda e: SolutionPeel().findRedundantConnection(e),
    "cycle_dfs": lambda e: SolutionCycleDFS().findRedundantConnection(e),
    "final": lambda e: Solution().findRedundantConnection(e),
}
all_methods = {"brute": find_redundant_bruteforce, **impls}

rng = random.Random(11)
for n in [100, 400, 1000]:
    edges = random_graph(rng, n)
    line = f"n = {n:4d}:"
    for name, fn in all_methods.items():
        t0 = time.perf_counter()
        fn(edges)
        line += f"  {name} {1000 * (time.perf_counter() - t0):7.2f}ms"
    print(line)

n =  100:  brute    0.05ms  uf    0.02ms  dfs    0.16ms  peel    0.04ms  cycle_dfs    0.04ms  final    0.02ms
n =  400:  brute    1.04ms  uf    0.11ms  dfs    2.38ms  peel    0.21ms  cycle_dfs    0.21ms  final    0.12ms
n = 1000:  brute   25.99ms  uf    0.32ms  dfs    5.86ms  peel    0.49ms  cycle_dfs    0.20ms  final    0.25ms


## 12. Bộ test

Vì bài có **ràng buộc "cạnh sau cùng"** nhưng cũng đủ khái niệm "nhiều cạnh xóa được", ta test bằng **hàm kiểm tra tính hợp lệ** `is_valid_answer` thay vì so với một đáp án cứng, **cộng thêm** so sánh trực tiếp các cách với oracle. Các phần: ví dụ đề, edge case, phản ví dụ, đối chiếu mọi cách trên nhiều đồ thị ngẫu nhiên có seed (đúng ràng buộc `a < b`, không cạnh lặp, liên thông), và ràng buộc lớn nhất `n = 1000` với nhiều hình dạng cây (đường, sao, ngẫu nhiên).

In [20]:
def check_all(edges, expected=None):
    results = {name: fn([e[:] for e in edges]) for name, fn in all_methods.items()}
    for name, r in results.items():
        assert is_valid_answer(edges, r), (name, r, edges)       # hợp lệ theo hàm kiểm tra
    assert len({tuple(r) for r in results.values()}) == 1, (edges, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (edges, r, expected)
    return r

# 1) Ví dụ đề
for edges, expected in examples:
    check_all(edges, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    ([[1, 2], [2, 3], [1, 3]], [1, 3]),                                # tam giác nhỏ nhất (n = 3)
    ([[1, 2], [1, 3], [2, 3]], [2, 3]),
    ([[1, 3], [2, 3], [1, 2]], [1, 2]),                                # chu trình khép ở cạnh cuối danh sách
    ([[1, 4], [3, 4], [1, 3], [1, 2], [4, 5]], [1, 3]),                # cạnh cầu đứng sau chu trình
    ([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]], [1, 4]),                # cạnh cầu [1,5] đứng sau nhưng không xóa được
    ([[1, 2], [1, 3], [1, 4], [1, 5], [2, 3]], [2, 3]),                # sao + 1 cạnh
    ([[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]], [1, 5]),                # chu trình độ dài 5 (cả đồ thị là một vòng)
    ([[1, 5], [1, 2], [2, 3], [3, 4], [4, 5]], [4, 5]),                # cùng vòng, thứ tự cạnh khác -> đáp án khác
]
for edges, expected in edge:
    check_all(edges, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


In [21]:
# 3) Phản ví dụ: có nhiều cạnh xóa được nhưng chỉ MỘT là đáp án hợp lệ (cạnh sau cùng)
edges = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
idx = removable_indices(edges)
print("các cạnh xóa được:", [edges[i] for i in idx])
for i in idx:
    print(f"  {edges[i]} hợp lệ theo hàm kiểm tra? {is_valid_answer(edges, edges[i])}")
assert [is_valid_answer(edges, edges[i]) for i in idx] == [False, False, False, True]

# 4) Đảo thứ tự cạnh trong cùng một đồ thị -> đáp án đổi theo "cạnh sau cùng của chu trình"
base = [[1, 2], [2, 3], [3, 4], [1, 4]]
seen_answers = set()
for perm in [base, base[::-1], [base[1], base[3], base[0], base[2]]]:
    ans = check_all(perm + [[1, 5]])
    seen_answers.add(tuple(ans))
    print(perm + [[1, 5]], "->", ans)
assert len(seen_answers) > 1

các cạnh xóa được: [[1, 2], [2, 3], [3, 4], [1, 4]]
  [1, 2] hợp lệ theo hàm kiểm tra? False
  [2, 3] hợp lệ theo hàm kiểm tra? False
  [3, 4] hợp lệ theo hàm kiểm tra? False
  [1, 4] hợp lệ theo hàm kiểm tra? True
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] -> [1, 4]
[[1, 4], [3, 4], [2, 3], [1, 2], [1, 5]] -> [1, 2]
[[2, 3], [1, 4], [1, 2], [3, 4], [1, 5]] -> [3, 4]


In [22]:
# 5) Ngẫu nhiên có seed: n nhỏ (3..12), so tất cả các cách + hàm kiểm tra
rng = random.Random(684)
for _ in range(500):
    n = rng.randint(3, 12)
    check_all(random_graph(rng, n))
print("ngẫu nhiên nhỏ: OK")

# 6) Ngẫu nhiên vừa (13..60) và lớn (nhiều hình dạng cây)
for _ in range(100):
    check_all(random_graph(rng, rng.randint(13, 60)))
print("ngẫu nhiên vừa: OK")

def shaped_graph(rng, n, shape):
    """Cây theo hình dạng: 'path', 'star', 'caterpillar', rồi thêm 1 cạnh; xáo cạnh; a < b."""
    edge_set = set()
    for v in range(2, n + 1):
        if shape == "path":
            u = v - 1
        elif shape == "star":
            u = 1
        else:
            u = max(1, v - rng.randint(1, 3))
        edge_set.add((u, v))
    while True:
        a, b = rng.sample(range(1, n + 1), 2)
        e = (min(a, b), max(a, b))
        if e not in edge_set:
            edge_set.add(e)
            break
    edges = [list(e) for e in edge_set]
    rng.shuffle(edges)
    return edges

for shape in ["path", "star", "caterpillar"]:
    for n in [3, 50, 1000]:
        e = shaped_graph(rng, n, shape)
        r = {name: fn([x[:] for x in e]) for name, fn in impls.items()}
        assert len({tuple(v) for v in r.values()}) == 1
        assert is_valid_answer(e, r["final"]) if n <= 200 else True
        print(f"hình {shape:11s} n = {n:4d}: đáp án {r['final']}")
big = random_graph(rng, 1000)
t0 = time.perf_counter()
r = {name: fn([x[:] for x in big]) for name, fn in impls.items()}
assert len({tuple(v) for v in r.values()}) == 1
assert is_valid_answer(big, r["final"])
print("n = 1000 ngẫu nhiên: 5 cách khớp và hợp lệ", f"({time.perf_counter() - t0:.2f}s)")

ngẫu nhiên nhỏ: OK


ngẫu nhiên vừa: OK


hình path        n =    3: đáp án [1, 2]
hình path        n =   50: đáp án [25, 26]
hình path        n = 1000: đáp án [384, 385]
hình star        n =    3: đáp án [1, 2]
hình star        n =   50: đáp án [1, 31]


hình star        n = 1000: đáp án [1, 33]
hình caterpillar n =    3: đáp án [1, 3]
hình caterpillar n =   50: đáp án [41, 42]
hình caterpillar n = 1000: đáp án [316, 318]
n = 1000 ngẫu nhiên: 5 cách khớp và hợp lệ (0.02s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả cạnh **đầu tiên** của chu trình thay vì cạnh sau cùng

Tìm được chu trình rồi trả cạnh nào đó "tìm thấy đầu tiên" (ví dụ theo thứ tự đầu vào từ trái).

In [23]:
class WrongFirstOfCycle:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        adj = [[] for _ in range(n + 1)]
        deg = [0] * (n + 1)
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
            deg[a] += 1
            deg[b] += 1
        removed = [False] * (n + 1)
        queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
        while queue:
            u = queue.popleft()
            removed[u] = True
            for v in adj[u]:
                if not removed[v]:
                    deg[v] -= 1
                    if deg[v] == 1:
                        queue.append(v)
        for a, b in edges:                           # SAI: duyệt từ đầu, không phải từ cuối
            if not removed[a] and not removed[b]:
                return [a, b]

ex = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]
print("sai :", WrongFirstOfCycle().findRedundantConnection(ex), "(đúng phải là [1, 4])")
print("đúng:", Solution().findRedundantConnection(ex))

sai : [1, 2] (đúng phải là [1, 4])
đúng: [1, 4]


**Vì sao sai:** tất cả cạnh trên chu trình đều xóa được, nhưng đề yêu cầu cạnh **sau cùng** theo thứ tự đầu vào. **Cách sửa:** duyệt `reversed(edges)` (hoặc dùng Union-Find, tự động trả cạnh sau cùng của chu trình).

### Lỗi 2: so sánh `p[a] == p[b]` mà không tìm **gốc**

In [24]:
class WrongNoFind:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))
        for a, b in edges:
            if p[a] == p[b]:                         # SAI: so cha trực tiếp, không phải gốc
                return [a, b]
            p[b] = a                                 # SAI: cũng không nối gốc
        return None

ex = [[1, 3], [2, 3], [2, 4], [1, 2]]
print("sai :", WrongNoFind().findRedundantConnection(ex), "(đúng phải là [1, 2])")
print("đúng:", Solution().findRedundantConnection(ex))

sai : None (đúng phải là [1, 2])
đúng: [1, 2]


**Vì sao sai:** "cha" trực tiếp không phản ánh tập mà nút thuộc về; hai nút cùng tập có thể có cha khác nhau. **Cách sửa:** luôn so sánh **gốc** `find(a) == find(b)` và nối **gốc với gốc**.

### Lỗi 3: mảng cha thiếu một phần tử vì nút đánh số từ 1

In [25]:
class WrongSize:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n))                           # SAI: thiếu chỉ số n, nút đánh số 1..n

        def find(x):
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:
                return [a, b]
            p[ra] = rb

ex = [[1, 2], [2, 3], [1, 3]]
try:
    print(WrongSize().findRedundantConnection(ex))
except IndexError as e:
    print("sai : IndexError:", e)
print("đúng:", Solution().findRedundantConnection(ex))

sai : IndexError: list index out of range
đúng: [1, 3]


**Vì sao sai:** nút `n` cần chỉ số `n` trong `p`, nhưng `list(range(n))` chỉ có `0..n-1`. **Cách sửa:** `list(range(n + 1))`, hoặc trừ 1 khi đọc (`find(a - 1)`).

### Lỗi 4: coi "cạnh có cả hai đầu bậc `>= 2`" là "cạnh trên chu trình"

In [26]:
class WrongDegree:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        deg = [0] * (n + 1)
        for a, b in edges:
            deg[a] += 1
            deg[b] += 1
        for a, b in reversed(edges):
            if deg[a] >= 2 and deg[b] >= 2:          # SAI: bậc ban đầu chưa đủ để kết luận nằm trên chu trình
                return [a, b]

ex = [[2, 3], [2, 5], [3, 5], [1, 2], [1, 4]]
print("sai :", WrongDegree().findRedundantConnection(ex), "(đúng phải là [3, 5])")
print("đúng:", Solution().findRedundantConnection(ex))

sai : [1, 2] (đúng phải là [3, 5])
đúng: [3, 5]


**Vì sao sai:** cạnh `[1,2]` nối nút `1` (bậc 2, vì còn nối nút `4`) và nút `2` (bậc 3), nhưng nó là **cầu**; nút `1` chỉ "bậc 2" nhờ nhánh lá `4`. **Cách sửa:** **cắt lá lặp lại** (cách 4) để loại hết cành, hoặc dùng Union-Find.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "cạnh thừa / chu trình trong đồ thị vô hướng":**

- "Cây + một cạnh" hoặc "cạnh nào làm xuất hiện chu trình đầu tiên" ⇒ **Union-Find** duyệt cạnh theo thứ tự: cạnh nào có hai đầu **đã cùng tập** là cạnh khép chu trình.
- Hỏi về **liên thông động** khi thêm cạnh dần (không xóa) ⇒ Union-Find là công cụ chuẩn.
- Cần "các nút nằm trên chu trình": **cắt lá** (topological pruning) hoặc DFS với cạnh ngược.

**Mẹo cài đặt:**

- Nút đánh số `1..n` ⇒ cấp `n + 1` ô, hoặc trừ 1.
- Viết `find` bằng **vòng lặp** (path halving) để khỏi lo đệ quy sâu.
- Khi đề có luật chọn đáp án ("nếu nhiều đáp án, trả cạnh sau cùng"), hãy xác định **cách duyệt** nào cho kết quả đúng luật tự nhiên (ở đây: duyệt theo thứ tự đầu vào).
- Viết **hàm kiểm tra tính hợp lệ** cho đáp án — rất hữu ích khi bài có nhiều đáp án đúng.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 685. Redundant Connection II | đồ thị **có hướng**: nút có hai cha, hoặc chu trình |
| 261. Graph Valid Tree | kiểm tra có là cây không (đủ `n-1` cạnh + không chu trình) |
| 547. Number of Provinces | đếm thành phần liên thông bằng Union-Find / DFS |
| 1319. Number of Operations to Make Network Connected | đếm cạnh thừa và thành phần |
| 1584. Min Cost to Connect All Points | cây khung nhỏ nhất (Kruskal dùng Union-Find) |

## 15. Biến thể

### 15.1. Liệt kê **mọi** cạnh có thể xóa

Mọi cạnh trên chu trình. Dùng cách cắt lá để tìm chu trình rồi lọc cạnh có cả hai đầu thuộc chu trình; kiểm bằng oracle `removable_indices`.

In [27]:
def all_removable_edges(edges):
    n = len(edges)
    adj = [[] for _ in range(n + 1)]
    deg = [0] * (n + 1)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
        deg[a] += 1
        deg[b] += 1
    removed = [False] * (n + 1)
    queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
    while queue:
        u = queue.popleft()
        removed[u] = True
        for v in adj[u]:
            if not removed[v]:
                deg[v] -= 1
                if deg[v] == 1:
                    queue.append(v)
    return [e for e in edges if not removed[e[0]] and not removed[e[1]]]

rng = random.Random(151)
for _ in range(300):
    e = random_graph(rng, rng.randint(3, 25))
    assert all_removable_edges(e) == [e[i] for i in removable_indices(e)]
print("ví dụ 2: các cạnh xóa được ->", all_removable_edges([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

ví dụ 2: các cạnh xóa được -> [[1, 2], [2, 3], [3, 4], [1, 4]]


### 15.2. Kiểm tra một đồ thị có là cây không (LeetCode 261)

Cho `n` nút và danh sách cạnh. Là cây ⇔ có **đúng `n - 1` cạnh** và **không có cạnh nào khép chu trình** (Union-Find). Kiểm bằng `is_tree` dựa trên BFS.

In [28]:
def valid_tree(n, edges):
    if len(edges) != n - 1:
        return False
    p = list(range(n + 1))

    def find(x):
        while p[x] != x:
            p[x] = p[p[x]]
            x = p[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        p[ra] = rb
    return True

rng = random.Random(152)
for _ in range(1000):
    n = rng.randint(1, 8)
    k = rng.choice([max(0, n - 2), n - 1, n])
    all_pairs = [[a, b] for a in range(1, n + 1) for b in range(a + 1, n + 1)]
    if k > len(all_pairs):
        continue
    e = rng.sample(all_pairs, k)
    assert valid_tree(n, e) == is_tree(n, e), (n, e)
print("valid_tree khớp is_tree trên 1000 đồ thị ngẫu nhiên: OK")
print(valid_tree(5, [[1, 2], [1, 3], [3, 4], [1, 5]]), valid_tree(5, [[1, 2], [2, 3], [1, 3], [4, 5]]))

valid_tree khớp is_tree trên 1000 đồ thị ngẫu nhiên: OK
True False


### 15.3. Redundant Connection II — đồ thị **có hướng** (LeetCode 685)

Cây có hướng gốc `r`: mọi nút (trừ gốc) có **đúng một cha**, và mọi nút đến được từ gốc. Thêm một cạnh có hướng thừa: có hai khả năng — (a) một nút có **hai cha**, hoặc (b) tạo **chu trình** (nút gốc nhận thêm cha). Cách giải chuẩn: tìm nút có hai cha ⇒ hai cạnh ứng viên `cand1` (xuất hiện trước), `cand2` (sau). Chạy Union-Find **bỏ qua `cand2`**: nếu vẫn thấy chu trình thì đáp án là `cand1` (nếu có hai cha) hoặc cạnh khép chu trình (nếu không có nút hai cha); nếu không có chu trình thì đáp án là `cand2`.

In [29]:
def find_redundant_directed(edges):
    n = len(edges)
    parent = [0] * (n + 1)
    cand1 = cand2 = None
    for a, b in edges:
        if parent[b]:
            cand1, cand2 = [parent[b], b], [a, b]       # nút b có hai cha
        else:
            parent[b] = a
    p = list(range(n + 1))

    def find(x):
        while p[x] != x:
            p[x] = p[p[x]]
            x = p[x]
        return x

    for a, b in edges:
        if cand2 is not None and [a, b] == cand2:
            continue                                      # tạm bỏ qua cạnh ứng viên sau
        ra, rb = find(a), find(b)
        if ra == rb:
            return cand1 if cand1 is not None else [a, b]
        p[rb] = ra                                        # nối con vào cha (có hướng)
    return cand2

def is_rooted_tree(n, edges):
    if len(edges) != n - 1:
        return False
    indeg = [0] * (n + 1)
    adj = [[] for _ in range(n + 1)]
    for a, b in edges:
        indeg[b] += 1
        adj[a].append(b)
    if any(indeg[v] > 1 for v in range(1, n + 1)):
        return False
    roots = [v for v in range(1, n + 1) if indeg[v] == 0]
    if len(roots) != 1:
        return False
    seen = {roots[0]}
    stack = [roots[0]]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen) == n

def directed_bruteforce(edges):
    n = len(edges)
    for i in range(n - 1, -1, -1):
        if is_rooted_tree(n, edges[:i] + edges[i + 1:]):
            return edges[i]

def random_directed(rng, n):
    labels = list(range(1, n + 1))
    rng.shuffle(labels)
    edge_set = set()
    for v in range(1, n):
        u = rng.randrange(v)
        edge_set.add((labels[u], labels[v]))
    while True:
        a, b = rng.sample(range(1, n + 1), 2)
        if (a, b) not in edge_set:
            edge_set.add((a, b))
            break
    edges = [list(e) for e in edge_set]
    rng.shuffle(edges)
    return edges

print(find_redundant_directed([[1, 2], [1, 3], [2, 3]]), find_redundant_directed([[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]]))
rng = random.Random(153)
for _ in range(3000):
    e = random_directed(rng, rng.randint(3, 8))
    assert find_redundant_directed(e) == directed_bruteforce(e), e
print("Redundant Connection II khớp vét cạn trên 3000 đồ thị có hướng ngẫu nhiên: OK")

[2, 3] [4, 1]


Redundant Connection II khớp vét cạn trên 3000 đồ thị có hướng ngẫu nhiên: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử xóa từng cạnh | từ cuối, xóa rồi kiểm "còn là cây" bằng BFS | `O(n²)` | `O(n)` | Làm **oracle**; bám sát định nghĩa |
| 2. Union-Find | duyệt cạnh theo thứ tự, hai đầu cùng tập ⇒ trả | `O(n · α(n))` | `O(n)` | **Nộp bài**: nhanh, ngắn, tự cho cạnh sau cùng |
| 3. Thêm dần + DFS | trước khi thêm, DFS hỏi "đã có đường đi `a → b` chưa" | `O(n²)` | `O(n)` | Dễ hiểu nhất, không cần cấu trúc đặc biệt |
| 4. Cắt lá | gỡ lá lặp lại để chỉ còn chu trình, lấy cạnh cuối thuộc chu trình | `O(n)` | `O(n)` | Cần **toàn bộ chu trình** / mọi cạnh xóa được |
| 5. DFS cạnh ngược | DFS ghi cha/độ sâu, cạnh ngược ⇒ leo cha lấy chu trình | `O(n)` | `O(n)` | Khi quen DFS đồ thị; cần chính đường chu trình |

Bốn cách đầu cho cùng đáp án trên mọi test; cách 4 và 5 còn **trả về toàn bộ chu trình**, hữu ích khi cần nhiều hơn một cạnh.

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, dự đoán rồi đối chiếu đáp án ở mục 19:

```python
Solution().findRedundantConnection([[1,2],[1,3],[2,3]])
Solution().findRedundantConnection([[1,2],[2,3],[1,3]])
Solution().findRedundantConnection([[1,4],[3,4],[1,3],[1,2],[4,5]])
Solution().findRedundantConnection([[1,2],[2,3],[3,4],[1,4],[1,5]])
```

### Bài 2 (tính tay)

Với `edges = [[1,2],[3,4],[2,3],[1,5],[4,5]]` (nút `1..5`), hãy chạy Union-Find bằng tay: ghi mảng cha `p[1..5]` sau mỗi cạnh (nối gốc của `a` vào gốc của `b`) và nêu cạnh bị trả về.

### Bài 3 (điền TODO)

Hoàn thành hàm `find` (có nén đường đi) và phần thân vòng lặp của Union-Find.

In [30]:
def find_redundant_todo(edges):
    n = len(edges)
    p = list(range(n + 1))

    def find(x):
        # TODO: trả về gốc của x, nén đường đi
        return None

    for a, b in edges:
        # TODO: nếu hai đầu cùng gốc thì trả cạnh [a, b]; ngược lại nối hai gốc
        pass
    return None
# Không gọi find_redundant_todo ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết `find_redundant_dfs(edges)` theo **Cách 3**: dựng đồ thị dần; trước khi thêm cạnh `[a, b]`, dùng DFS kiểm tra `b` có tới được từ `a` không.

In [31]:
def find_redundant_dfs(edges):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `graph_has_unique_cycle(n, edges)` cho đồ thị **liên thông** `n` nút `n` cạnh (giả sử đầu vào đúng dạng của bài này) trả về **danh sách các nút trên chu trình** (sắp tăng dần). Kiểm chứng bằng định nghĩa: nút thuộc chu trình ⇔ là đầu mút của một cạnh xóa được.

In [32]:
def graph_has_unique_cycle(n, edges):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: hãy tìm chu trình trong đồ thị, rồi xem cạnh nào của chu trình đứng sau cùng.
- Mức 2: `[[1,4],[3,4],[1,3],[1,2],[4,5]]`: chu trình `1-4-3-1` gồm ba cạnh đầu; cạnh `[1,2]` và `[4,5]` là cầu.
- Mức 3: đáp án `[2,3]`, `[1,3]`, `[1,3]`, `[1,4]`.

**Bài 2**

- Mức 1: bắt đầu `p = [1,2,3,4,5]`; xét `[1,2]` nối gốc `1` vào gốc `2` (`p[1] = 2`).
- Mức 2: `[3,4]` cho `p[3] = 4`; `[2,3]`: `find(2) = 2`, `find(3) = 4` ⇒ nối `p[2] = 4`.
- Mức 3: `[1,5]`: `find(1) = 4` (qua `1→2→4`), `find(5) = 5` ⇒ nối `p[4] = 5`; `[4,5]`: `find(4) = 5` và `find(5) = 5` ⇒ cùng tập ⇒ trả `[4,5]`.

**Bài 3**

- Mức 1: `find` đi theo cha cho tới khi `p[x] == x`.
- Mức 2: nén đường đi: `p[x] = p[p[x]]` trong vòng lặp.
- Mức 3: `ra, rb = find(a), find(b)`; `if ra == rb: return [a, b]`; `p[ra] = rb`.

**Bài 4**

- Mức 1: giữ `adj`; trước khi `append` cạnh mới thì hỏi `b` có tới được từ `a`.
- Mức 2: kiểm tra bằng ngăn xếp + tập `seen` (mục 3.3).
- Mức 3: `if connected(a, b): return [a, b]`, rồi mới `adj[a].append(b); adj[b].append(a)`.

**Bài 5**

- Mức 1: cắt lá cho tới khi chỉ còn nút bậc `>= 2` mà không gỡ được nữa.
- Mức 2: nút chưa bị gỡ chính là nút trên chu trình.
- Mức 3: dùng `deque` các nút bậc `1`; kết quả `sorted(u for u in range(1, n + 1) if not removed[u])`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
answers1 = [
    ([[1, 2], [1, 3], [2, 3]], [2, 3]),
    ([[1, 2], [2, 3], [1, 3]], [1, 3]),
    ([[1, 4], [3, 4], [1, 3], [1, 2], [4, 5]], [1, 3]),
    ([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]], [1, 4]),
]
for edges, expected in answers1:
    got = Solution().findRedundantConnection(edges)
    assert got == expected and is_valid_answer(edges, got)
    print(edges, "->", got)

# Bài 2
print("Bài 2:")
res = trace_union_find([[1, 2], [3, 4], [2, 3], [1, 5], [4, 5]])
assert res == [4, 5]

[[1, 2], [1, 3], [2, 3]] -> [2, 3]
[[1, 2], [2, 3], [1, 3]] -> [1, 3]
[[1, 4], [3, 4], [1, 3], [1, 2], [4, 5]] -> [1, 3]
[[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]] -> [1, 4]
Bài 2:
cạnh [1, 2]: find(1) = 1 (đường [1]), find(2) = 2 (đường [2])
  -> khác tập: nối gốc 1 vào gốc 2; p[1..5] = [2, 2, 3, 4, 5]
cạnh [3, 4]: find(3) = 3 (đường [3]), find(4) = 4 (đường [4])
  -> khác tập: nối gốc 3 vào gốc 4; p[1..5] = [2, 2, 4, 4, 5]
cạnh [2, 3]: find(2) = 2 (đường [2]), find(3) = 4 (đường [3, 4])
  -> khác tập: nối gốc 2 vào gốc 4; p[1..5] = [2, 4, 4, 4, 5]
cạnh [1, 5]: find(1) = 4 (đường [1, 2, 4]), find(5) = 5 (đường [5])
  -> khác tập: nối gốc 4 vào gốc 5; p[1..5] = [2, 4, 4, 5, 5]
cạnh [4, 5]: find(4) = 5 (đường [4, 5]), find(5) = 5 (đường [5])
  -> cùng tập: cạnh [4, 5] khép chu trình, trả về


In [34]:
# Bài 3
def find_redundant_fill_answer(edges):
    n = len(edges)
    p = list(range(n + 1))

    def find(x):
        while p[x] != x:
            p[x] = p[p[x]]
            x = p[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]
        p[ra] = rb
    return None

# Bài 4
def find_redundant_dfs_answer(edges):
    n = len(edges)
    adj = [[] for _ in range(n + 1)]

    def connected(a, b):
        stack, seen = [a], {a}
        while stack:
            u = stack.pop()
            if u == b:
                return True
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        return False

    for a, b in edges:
        if connected(a, b):
            return [a, b]
        adj[a].append(b)
        adj[b].append(a)
    return None

# Bài 5
def graph_has_unique_cycle_answer(n, edges):
    adj = [[] for _ in range(n + 1)]
    deg = [0] * (n + 1)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
        deg[a] += 1
        deg[b] += 1
    removed = [False] * (n + 1)
    queue = deque(u for u in range(1, n + 1) if deg[u] == 1)
    while queue:
        u = queue.popleft()
        removed[u] = True
        for v in adj[u]:
            if not removed[v]:
                deg[v] -= 1
                if deg[v] == 1:
                    queue.append(v)
    return sorted(u for u in range(1, n + 1) if not removed[u])

rng = random.Random(1999)
for _ in range(300):
    n = rng.randint(3, 25)
    e = random_graph(rng, n)
    expected = Solution().findRedundantConnection(e)
    assert find_redundant_fill_answer(e) == expected
    assert find_redundant_dfs_answer(e) == expected
    on_cycle = sorted({x for ed in all_removable_edges(e) for x in ed})
    assert graph_has_unique_cycle_answer(n, e) == on_cycle
print("Bài 3, 4, 5: OK")
print("ví dụ Bài 5: chu trình của ví dụ 2 gồm các nút", graph_has_unique_cycle_answer(5, [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

Bài 3, 4, 5: OK
ví dụ Bài 5: chu trình của ví dụ 2 gồm các nút [1, 2, 3, 4]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (Union-Find duyệt cạnh theo thứ tự):**

```python
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        p = list(range(n + 1))                 # nút 1..n  ->  n + 1 ô

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]                 # nén đường đi
                x = p[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra == rb:                       # hai đầu đã cùng tập -> cạnh khép chu trình
                return [a, b]
            p[ra] = rb                         # nối GỐC với GỐC
        return []
```

**Câu thần chú:** *"Cạnh nào có hai đầu đã cùng gốc thì cạnh đó thừa; so gốc, nối gốc, `n + 1` ô."*

**Checklist trước khi submit:**

- [ ] Hiểu: đồ thị `n` nút `n` cạnh liên thông ⇒ **đúng một chu trình**; mọi cạnh trên chu trình xóa được.
- [ ] Đề yêu cầu cạnh **sau cùng**: duyệt cạnh **theo thứ tự đầu vào** (Union-Find) hoặc `reversed(edges)` (cách cắt lá).
- [ ] So sánh **gốc** (`find`), không so cha trực tiếp; nối **gốc với gốc**.
- [ ] Mảng cha có `n + 1` ô (nút đánh số từ `1`).
- [ ] `find` viết lặp, nén đường đi.
- [ ] Đã thử: tam giác `n = 3`, chu trình là cả đồ thị, cạnh cầu đứng sau chu trình, thứ tự cạnh đảo.
- [ ] Đã kiểm bằng hàm kiểm tra tính hợp lệ (xóa còn là cây, và là cạnh sau cùng thỏa điều kiện).
- [ ] Độ phức tạp: gần `O(n)` thời gian, `O(n)` bộ nhớ.